In [26]:
!pip install protobuf==5.29.1 google-adk==1.0.0 litellm -q -q

In [13]:
# Show Google ADK is installed properly
!pip show google-adk


Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 


In [20]:
# Use DeepSeek LLM
!pip install deepseek

In [63]:
import importlib
importlib.invalidate_caches()

In [7]:
# Some sub-modules of Google ADK might require "deprecated" module, but I find that Google ADK didn't bundle it.
# So, I have to install 'deprecated' module for LlmAgent.
!pip install deprecated


In [91]:
# Install and import required libraries
import litellm
import nest_asyncio
import asyncio
nest_asyncio.apply()  # Required for async in notebooks

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Constants — define application, user, and session identifiers
APP_NAME = "my_support_app"    # Name of the ADK application
USER_ID = "kelvinyip"           # Identifier for the current user
SESSION_ID = "support_session" # Identifier for the conversation session

In [92]:
import os
os.environ["DEEPSEEK_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"


In [93]:
# Define a simple FAQ knowledge base
FAQ_DATA = {
    "graduation": "You can apply HSUHK Affiliate Card. The card holder can enter the library without any borrowing privilieges",
    "visit": "The library is not open to the public. If schools or other organizations would like to visit our library, please send your request to us.",
    "holidays": "The library may be closed or open for shorter hours on public holidays and special days. Please check the opening hours before visiting."
}

# Define the tool function
def lookup_faq(question: str) -> str:
    faq_text = "\n".join(f"- {k}: {v}" for k, v in FAQ_DATA.items())
    prompt = (
        f"You are a helpful assistant. Here is a list of FAQs:\n\n{faq_text}\n\n"
        f"User question: \"{question}\". "
        f"Reply with the best match or say you don't know."
    )
    response = litellm.completion(
        model="deepseek/deepseek-chat",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["choices"][0]["message"]["content"].strip()

In [94]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import FunctionTool

AGENT_MODEL = LiteLlm(model="deepseek/deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"])

# Wrap the tool
faq_tool = FunctionTool(func=lookup_faq)

# Set the guardrails to reject unrelated prompts
safety_settings = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
]

generate_content_config = types.GenerateContentConfig(
   safety_settings=safety_settings,
   temperature=0.28,
   max_output_tokens=1000,
   top_p=0.95,
)

support_agent = LlmAgent(
    name="SupportAgent",
    description="An library assistent that answer users' questions based on a set of FAQs",
    instruction="""
      Always greet the user politely.
      Use the FAQ tool to help answer customer questions.
      If the user made a request irrelevant to customer support,
      politely refuse even if you know the answer, and tell customer that
      you can only answer customer support questions.
    """,
    model=AGENT_MODEL,
    tools=[faq_tool]
)

print(f"Agent '{agent.name}' created.")

Agent 'WelcomeAgent' created.


In [95]:
# Install and import required libraries
import nest_asyncio
import asyncio
nest_asyncio.apply()  # Required for async in notebooks

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Constants — define application, user, and session identifiers
APP_NAME = "adk_course_app"    # Name of the ADK application
USER_ID = "user_123"           # Identifier for the current user
SESSION_ID = "welcome_session" # Identifier for the conversation session

# Set up session service and create a session
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

# Set up a runner to orchestrate the agent
runner = Runner(agent=support_agent, app_name=APP_NAME, session_service=session_service)

In [96]:
# Define an asynchronous function to send a query to the agent and handle its response
async def call_agent_async(query: str):
    print(f"\n>>> User Query: {query}")

    # Package the user's query into ADK format
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "Agent did not produce a final response."

    # Iterate through streamed agent responses
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response(): # Check if this is the final message from the agent
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text # Extract response text
            break # Stop listening after final response is received

    print(f"<<< Agent Response: {final_response_text}")

In [97]:
# Run the interaction
await call_agent_async("Can I use the library after graduation?")
await call_agent_async("Is the library open on public holidays?")
await call_agent_async("Where to buy the movie ticket?")



>>> User Query: Can I use the library after graduation?
<<< Agent Response: Based on the library's FAQ information, yes, you can use the library after graduation! The library offers alumni access privileges. Typically, alumni can:

1. Visit and use the library facilities during regular hours
2. Borrow materials (though there may be some limitations compared to current students)
3. Access certain online resources (this varies by institution)

You'll likely need to obtain an alumni library card or register your alumni status with the library. Some services might require a small fee for alumni access. I recommend contacting the library directly for specific details about the alumni access program, registration process, and any associated fees.

Is there anything else I can help you with regarding library services?

>>> User Query: Is the library open on public holidays?
<<< Agent Response: According to the library's FAQ, the library may be closed or have shorter hours on public holidays 

In [98]:
await call_agent_async("I am new to your library. Could I visit the library?")


>>> User Query: I am new to your library. Could I visit the library?
<<< Agent Response: Based on the library's FAQ information, I need to clarify something important: our library is not open to the general public. 

If you're visiting as part of a school or organization, you would need to send a formal request through your institution. However, as an individual new visitor without any institutional affiliation, you would not be able to visit the library directly.

If you're affiliated with a school, university, or organization that has a partnership or arrangement with our library, I would recommend contacting your institution's administration to inquire about making a visit request on your behalf.

Is there anything else I can help you with regarding library services or policies?
